# Building Shor's Algorithm from Scratch, Part 1: The Quantum Adder

[Another tutorial](400_shor.ipynb) in this collection already walks through the *theory* of Shor's algorithm end to end. This is the start of a companion series that builds a **working implementation** up from first principles, one arithmetic building block at a time: quantum adder → modulo adder → controlled modular multiplier → modular exponentiation → the full algorithm. Each part will be added as it's ready.

This tutorial covers the first, most basic piece: a quantum circuit that adds two binary numbers.

Reference: V. Vedral, A. Barenco, A. Ekert, ["Quantum Networks for Elementary Arithmetic Operations"](https://arxiv.org/abs/quant-ph/9511018) (1995).

## Why an adder?

Recall the shape of Shor's algorithm (see [400_shor.ipynb](400_shor.ipynb) for the full derivation): the quantum part finds the *order* $r$ of some $x$ modulo $N$, i.e. the smallest $r$ with $x^r \equiv 1 \pmod N$. Finding $r$ boils down to computing $x^j \bmod N$ in superposition over many values of $j$ at once — **modular exponentiation**. Modular exponentiation is built out of repeated **controlled modular multiplication**, which is itself built out of repeated **modular addition**. So the very bottom of the stack is: how do you add two numbers on a quantum computer?

## From classical gates to quantum gates

Three quantum gates are enough to reproduce any classical (reversible) logic:

| Classical operation | Quantum gate |
|:---|:---|
| NOT | `X` |
| XOR | `CX` (CNOT) |
| AND | `CCX` (Toffoli) |

`CX[a, b]` flips qubit `b` if qubit `a` is `1` — that's exactly `b := b XOR a`. `CCX[a, b, c]` flips `c` only when *both* `a` and `b` are `1` — that's `c := c OR (a AND b)` when `c` starts at `0`. Ordinary binary addition of two bits is built from exactly these two operations: a carry-out bit is the AND of the inputs (plus the incoming carry), and the sum bit is the XOR of the inputs (plus the incoming carry).

## CARRY and SUM

For one bit position, with incoming carry $c$, addends $a$, $b$, and outgoing carry $c_{next}$:

```
CARRY: c,a,b,c_next -> c,a,b, c_next XOR (carry-out of a+b+c)
SUM:   c,a,b         -> c,a, (a XOR b XOR c)
```

Chaining `CARRY` from the least significant bit upward produces every carry bit; `SUM` then turns each carry into the corresponding sum bit. Because quantum gates must be reversible, `CARRY` and `SUM` are literally just circuits made of `CCX`/`CX`, so they're automatically invertible — we'll need `CARRY`'s inverse (`CARRY_INV`) further down.

In [1]:
!pip install git+https://github.com/blueqat/blueqatSDK

/Users/yuichirominato/.pyenv/shims/pip: line 8: /opt/homebrew/opt/pyenv/bin/pyenv: No such file or directory


In [2]:
from blueqat import Circuit

def carry(c, a, b, c_next):
    """CCX[a,b,c_next].CX[a,b].CCX[c,b,c_next] -- propagates the carry out of bit position a+b+c into c_next."""
    circ = Circuit()
    circ.ccx[a, b, c_next]
    circ.cx[a, b]
    circ.ccx[c, b, c_next]
    return circ

def carry_inv(c, a, b, c_next):
    """The inverse of carry(): same three (self-inverse) gates, applied in reverse order."""
    circ = Circuit()
    circ.ccx[c, b, c_next]
    circ.cx[a, b]
    circ.ccx[a, b, c_next]
    return circ

def sum_gate(c, a, b):
    """CX[a,b].CX[c,b] -- turns b into a XOR b XOR c (the sum bit), leaving a and c unchanged."""
    circ = Circuit()
    circ.cx[a, b]
    circ.cx[c, b]
    return circ

## Assembling the full adder

To add two $n$-bit numbers $a$ and $b$ we need $3n+1$ qubits, split into three registers:

- `c[0..n-1]`: carry register, starts at $\ket{0}$
- `a[0..n-1]`: first addend
- `b[0..n]`: second addend, **one bit wider than the inputs** — the sum $a+b$ can need one extra bit, and this is where the answer ends up

The circuit runs `carry` from the low bit upward to ripple the carry all the way to the top (into `b[n]`, the extra bit), applies `sum_gate` to the top bit, and then walks back down applying `carry_inv` followed by `sum_gate` at each lower bit — computing every remaining sum bit while simultaneously **uncomputing** the carry register back to all zeros so it can be reused or discarded cleanly.

In [3]:
def build_adder(input_a, input_b, n):
    """n-bit ripple-carry adder. Returns (circuit, b_register_qubit_indices)."""
    num_qubits = 3 * n + 1
    circ = Circuit(num_qubits)

    c = list(range(n))
    a = list(range(n, 2 * n))
    b = list(range(2 * n, 3 * n + 1))

    # Encode the two inputs (c stays at |0...0>)
    for i in range(n):
        if (input_a >> i) & 1:
            circ.x[a[i]]
        if (input_b >> i) & 1:
            circ.x[b[i]]

    # Ripple the carry up
    for i in range(n - 1):
        circ += carry(c[i], a[i], b[i], c[i + 1])
    circ += carry(c[n - 1], a[n - 1], b[n - 1], b[n])

    # Top sum bit
    circ.cx[a[n - 1], b[n - 1]]
    circ += sum_gate(c[n - 1], a[n - 1], b[n - 1])

    # Walk back down: uncompute the carries, filling in the remaining sum bits
    for i in range(n - 2, -1, -1):
        circ += carry_inv(c[i], a[i], b[i], c[i + 1])
        circ += sum_gate(c[i], a[i], b[i])

    return circ, b

## Reading out the answer

**A note on bit order**: this SDK's `.run(shots=...)` returns bitstrings with qubit 0 as the *rightmost* character (standard binary-string convention), so to read off qubit `q`'s value from a result string of length `N` we index `state[N - 1 - q]`. `b[0]` is the least significant bit of the sum, so we read the `b` register from `b[0]` (least significant) up to `b[n]` (most significant) and assemble the integer accordingly.

Because this circuit is built entirely from `X`/`CX`/`CCX` — gates that just flip qubits around rather than creating superposition — a single deterministic input always produces a single deterministic output. `shots=1` is enough to get an exact answer every time.

In [4]:
def run_adder(val_a, val_b):
    n = max(val_a, val_b, 1).bit_length()
    circuit, b = build_adder(val_a, val_b, n)
    state = circuit.m[:].run(shots=1).most_common(1)[0][0]
    total_qubits = len(state)
    bits = [state[total_qubits - 1 - q] for q in b]  # bits[0] = LSB
    return int("".join(reversed(bits)), 2)

In [5]:
for x, y in [(10, 6), (1, 1), (3, 5), (7, 7), (0, 5), (15, 15)]:
    result = run_adder(x, y)
    print(f"{x} + {y} = {result}  (expected {x + y}, {'OK' if result == x + y else 'MISMATCH'})")

10 + 6 = 16  (expected 16, OK)
1 + 1 = 2  (expected 2, OK)
3 + 5 = 8  (expected 8, OK)
7 + 7 = 14  (expected 14, OK)
0 + 5 = 5  (expected 5, OK)
15 + 15 = 30  (expected 30, OK)


All six cases match ordinary integer addition exactly, as expected for a purely reversible-logic circuit.

## What's next

This adder is the innermost building block. The next tutorial in the series adds **modular** addition ($a+b \bmod N$) on top of it — the actual operation Shor's algorithm needs, since every intermediate value in modular exponentiation has to stay within $[0, N)$.